In [7]:
import pandas as pd
import re

df = pd.read_csv("../data/movie_reviews/IMDB_Dataset.csv")

def clean_text(text):
    text = re.sub(r'<.*?>', '', text)
    text = text.lower()
    return text

df['review'] = df['review'].apply(clean_text)
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})

print(df.head())
print(df['sentiment'].value_counts())


                                              review  sentiment
0  one of the other reviewers has mentioned that ...          1
1  a wonderful little production. the filming tec...          1
2  i thought this was a wonderful way to spend ti...          1
3  basically there's a family where a little boy ...          0
4  petter mattei's "love in the time of money" is...          1
sentiment
1    25000
0    25000
Name: count, dtype: int64


In [8]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import numpy as np

max_words = 15000
max_len = 250

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(df['review'])

sequences = tokenizer.texts_to_sequences(df['review'])
X = pad_sequences(sequences, maxlen=max_len)

y = np.array(df['sentiment'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


Train shape: (40000, 250)
Test shape: (10000, 250)


In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional

model = Sequential()

model.add(Embedding(input_dim=max_words, output_dim=128, input_length=max_len))
model.add(Bidirectional(LSTM(128)))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()


c:\python notebooks\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [10]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop]
)


Epoch 1/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 165s 327ms/step - accuracy: 0.7697 - loss: 0.4794 - val_accuracy: 0.6475 - val_loss: 0.6253
Epoch 2/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 191s 381ms/step - accuracy: 0.8344 - loss: 0.3908 - val_accuracy: 0.8622 - val_loss: 0.3252
Epoch 3/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 199s 398ms/step - accuracy: 0.9098 - loss: 0.2352 - val_accuracy: 0.8780 - val_loss: 0.3171
Epoch 4/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 210s 420ms/step - accuracy: 0.9384 - loss: 0.1702 - val_accuracy: 0.8756 - val_loss: 0.3118
Epoch 5/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 225s 449ms/step - accuracy: 0.9584 - loss: 0.1210 - val_accuracy: 0.8859 - val_loss: 0.3389
Epoch 6/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 219s 438ms/step - accuracy: 0.9710 - loss: 0.0908 - val_accuracy: 0.8706 - val_loss: 0.3802


In [11]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Final Test Accuracy:", accuracy)


313/313 ━━━━━━━━━━━━━━━━━━━━ 13s 40ms/step - accuracy: 0.8822 - loss: 0.3018
Final Test Accuracy: 0.8822000026702881


In [12]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = model.predict(X_test)
y_pred = (y_pred > 0.5).astype(int)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


313/313 ━━━━━━━━━━━━━━━━━━━━ 15s 47ms/step
[[4537  424]
 [ 754 4285]]
              precision    recall  f1-score   support

           0       0.86      0.91      0.89      4961
           1       0.91      0.85      0.88      5039

    accuracy                           0.88     10000
   macro avg       0.88      0.88      0.88     10000
weighted avg       0.88      0.88      0.88     10000



In [14]:
import pickle

model.save("../models/lstm_imdb_sentiment.keras")

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)
